# Общие функции

In [ ]:
import pandas as pd
import openpyxl
import numpy as np

In [ ]:
# загружаю названия регионов с кодами ОКАТО 
#get spreadsheets key from url
gsheetkey = "your/key"

#sheet name
sheet_name = 'список регионов sorted'

url=f'your/sheet/ccc?key={gsheetkey}&output=xlsx'
lookup = pd.read_excel(url,sheet_name=sheet_name)

In [ ]:
# функция для присваивания ОКАТО регионам по их названиям

def vlookup_okato(df, lookup_df, region_col):
    mapping = (
        lookup_df.dropna(subset=[region_col])
        .drop_duplicates(subset=[region_col])
        .set_index(region_col)['object_okato']
        .astype(str)
    )
    # df['Номер региона (ОКАТО)'] = df['Название региона'].map(mapping)
    new_col = df['Название региона'].map(mapping)
    df.insert(loc = 0,
          column = 'Номер региона (ОКАТО)',
          value = new_col)
    return df

In [ ]:
# присваиваю ОКАТО по кодам регионов КОУЖ

def vlookup_kouzh(df, lookup_df, region_col):
    mapping = (
        lookup_df.dropna(subset=[region_col])
        .drop_duplicates(subset=[region_col])
        .set_index(region_col)['object_okato']
        .astype(str)
    )
    new_col = df['Номер региона'].map(mapping)
    df.insert(loc = 0,
          column = 'Номер региона (ОКАТО)',
          value = new_col)
    return df

In [ ]:
# присваиваю ОКАТО по названию регионов

def okato_to_region(df, lookup_df, region_col):
    lookup_df = lookup_df[['object_okato', region_col]].copy()
    lookup_df[region_col] = lookup_df[region_col].astype(str)
    # lookup_df['object_okato'] = _okato_key(lookup_df['object_okato'])
    lookup_df['object_okato'] = lookup_df['object_okato'].astype(str).str.replace('.0', '', regex=False).str.strip()

    lookup_df = lookup_df.sort_values(by=region_col, na_position='last').drop_duplicates('object_okato')

    df = df.copy()
    df['Номер региона (ОКАТО)'] = df['Номер региона (ОКАТО)'].astype(str).str.replace('.0', '', regex=False).str.strip()

    mapping = dict(zip(lookup_df['object_okato'], lookup_df[region_col]))
    df.insert(loc=0, column='Название региона', value=df['Номер региона (ОКАТО)'].map(mapping))
    return df

In [ ]:
year_cols = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Обработка данных для обновления

### Средняя начисленная зарплата по регионам 2019, 2021, 2023 годы

In [ ]:
# достаю данные о зарплатах за 2019 год
salary = pd.read_excel('your/path/to/data/sved_57-t_2019/Статбюлетень 2019 - new/Часть 2/Таб. 40.xlsx')

salary = salary.rename(columns={
                    'Unnamed: 1': 'Средняя заработная плата по группам занятий, все работники',
                    'Unnamed: 2': 'Средняя заработная плата по группам занятий, мужчины',
                    'Unnamed: 3': 'Средняя заработная плата по группам занятий, женщины',
                    '40. Средняя начисленная заработная плата работников списочного состава по полу и \nсубъектам Российской Федерации за октябрь 2019 года ': 'Название региона'
                    })
salary['Название региона'] = salary['Название региона'].str.strip()
salary['Название региона'] = salary['Название региона'].str.replace('\n', ' ')
salary['Название региона'] = salary['Название региона'].str.replace('H', 'Н') # заменяю латинскую H (в Hижний Новгород) на кириллическую Н

salary = salary.drop(columns=[
                            # 'Unnamed: 1', 
                                'Unnamed: 4'
                                ])
salary = salary.drop(index=[0, 1, 2, 3])
salary = salary.drop_duplicates(subset=['Название региона'], keep='last') # удаляю дублирующейся Забайкальский край
# salary.tail()

# считаю гендерный разрыв в оплате труда, как у «Если быть точным» (ЕБТ)
salary['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)'] = ((salary['Средняя заработная плата по группам занятий, мужчины'] - 
                                                                                salary['Средняя заработная плата по группам занятий, женщины'])/
                                                                                salary['Средняя заработная плата по группам занятий, мужчины']*100)

# подгоняю данные под формат датасета ЕБТ
salary_long_19 = salary.melt(id_vars=['Название региона'], 
                        value_vars=['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', 
                                    'Средняя заработная плата по группам занятий, все работники', 
                                    'Средняя заработная плата по группам занятий, мужчины', 
                                    'Средняя заработная плата по группам занятий, женщины'], 
                        var_name='Название показателя', value_name=str(2019))

salary_long_19['Номер показателя (ID)'] = salary_long_19['Название показателя'].map({
                                                'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)': 95799, 
                                                'Средняя заработная плата по группам занятий, мужчины': 95789, 
                                                'Средняя заработная плата по группам занятий, все работники': 95916, 
                                                'Средняя заработная плата по группам занятий, женщины': 95790})

salary_long_19 = vlookup_okato(salary_long_19, lookup, 'salary_long')
salary_long_19.drop(salary_long_19[salary_long_19['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

salary_long_19 = salary_long_19[['Номер региона (ОКАТО)', 
                                #  'Название региона', 
                                 'Номер показателя (ID)', 
                                 '2019']]

In [ ]:
# функция для выделения и подсчета данных о 
# средней начисленной зарплате по полу
def extract_salary(year):
    # загружаю и обрабатываю данные
    salary = pd.read_excel(f'/your/path/to/data/sved_57-t_{year}/Статбюлетень {year}.xlsx', sheet_name='42')
    salary = salary.rename(columns={
                        'Unnamed: 1': 'Средняя заработная плата по группам занятий, все работники',
                        'Unnamed: 2': 'Средняя заработная плата по группам занятий, мужчины',
                        'Unnamed: 3': 'Средняя заработная плата по группам занятий, женщины',
                        f'42. Средняя начисленная заработная плата работников списочного состава по полу и \nсубъектам Российской Федерации за октябрь {year} года ': 'Название региона'
                        })
    salary['Название региона'] = salary['Название региона'].str.strip()
    salary['Название региона'] = salary['Название региона'].str.replace('\n', ' ')
    salary['Название региона'] = salary['Название региона'].str.replace('H', 'Н') # replace Latin H in Hижний Новгород with Cyrillic Н
    salary = salary.drop(columns=[
                                # 'Unnamed: 1', 
                                  'Unnamed: 4'
                                  ])
    salary = salary.drop(index=[0, 1, 2])
    # salary.head()

    # считаю гендерный разрыв в оплате труда, как у «Если быть точным» (ЕБТ)
    salary['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)'] = ((salary['Средняя заработная плата по группам занятий, мужчины'] - 
                                                                                    salary['Средняя заработная плата по группам занятий, женщины'])/
                                                                                    salary['Средняя заработная плата по группам занятий, мужчины']*100)

    # подгоняю данные под формат датасета ЕБТ
    salary_long_21 = salary.melt(id_vars=['Название региона'], 
                            value_vars=['Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', 
                                        'Средняя заработная плата по группам занятий, все работники', 
                                        'Средняя заработная плата по группам занятий, мужчины', 
                                        'Средняя заработная плата по группам занятий, женщины'], 
                            var_name='Название показателя', value_name=str(year))

    salary_long_21['Номер показателя (ID)'] = salary_long_21['Название показателя'].map({
                                                    'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)': 95799, 
                                                    'Средняя заработная плата по группам занятий, мужчины': 95789, 
                                                    'Средняя заработная плата по группам занятий, все работники': 95916, 
                                                    'Средняя заработная плата по группам занятий, женщины': 95790})

    salary_long_21 = vlookup_okato(salary_long_21, lookup, 'salary_long')
    salary_long_21.drop(salary_long_21[salary_long_21['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
    return salary_long_21

In [ ]:
# подсчет данных по 2021 году
salary_long_21 = extract_salary(2021)
salary_long_21 = salary_long_21[['Номер региона (ОКАТО)', 
                                #  'Название региона', 
                                 'Номер показателя (ID)', 
                                 '2021']]

# объединение данные 2019 и 2021 года
salary_long = salary_long_19.merge(
                                salary_long_21[['Номер региона (ОКАТО)', 'Номер показателя (ID)', '2021']], 
                                on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'], 
                                how='left')
# salary_long.sort_values(by='Номер региона (ОКАТО)').head(20)

# подсчет данных по 2023 году
salary_long_23 = extract_salary(2023)
salary_long = salary_long.merge(
                                salary_long_23[['Номер региона (ОКАТО)', 'Номер показателя (ID)', '2023']], 
                                on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'], 
                                how='left')

for c in ['2019', '2021', '2023']:
    salary_long[c] = pd.to_numeric(salary_long[c], errors='coerce').round(2)

salary_long.sort_values(by='Номер региона (ОКАТО)').head(20)

### Уход за детьми, «Комплексное наблюдение условий жизни населения»

In [ ]:
# функция для подсчета доли женщин, ухаживающих за детьми
# данные Комплексного наблюдения условий жизни населения (КОУЖ)
def calculate_childcare_share(group):
    answer = (group[group['I05_41'].isin([1, 2, 3, 4])]['KVZV'].sum()
                / group['KVZV'].sum()*100)
    return answer.round(0)

In [ ]:
# функция для подсчета доли женщин, ухаживающих за детьми, в указанном году
def get_kouzh(year):
    kouzh = pd.read_csv(
        f'your/path/to/data/КОУЖ (Комплексное наблюдение условий жизни населения)/{year}/КОУЖ_{year}_Датасет_по_индивидуальным_данным_основной.csv',
        sep=';')

    kouzh_zh = kouzh[kouzh['H01_01'] == 2] # выделить из общего датасета только женщин

    share_by_region = kouzh_zh.groupby('H00_02').apply(calculate_childcare_share)

    # make a dataset out of share_by_region
    share_by_region = share_by_region.reset_index()
    share_by_region = share_by_region.rename(columns={0: f'{year}',
                                                            'H00_02': 'Номер региона'})

    # добавим подсчет по всей стране
    answer = round(((kouzh_zh[kouzh_zh['I05_41'].isin([1, 2, 3, 4])]['KVZV'].sum()
                    / kouzh_zh['KVZV'].sum()*100)), 0)
    share_by_region = pd.concat([share_by_region, pd.DataFrame({'Номер региона': [100], f'{year}': [answer]})], ignore_index=True)
    return share_by_region

In [ ]:
year = 2020
kouzh = pd.read_csv(
    f'your/path/to/data/КОУЖ (Комплексное наблюдение условий жизни населения)/{year}/КОУЖ_{year}_Датасет_по_индивидуальным_данным_основной.csv',
    sep=';')
kouzh_zh = kouzh[kouzh['H01_01'] == 2]
answer = round((kouzh_zh[kouzh_zh['I05_41'].isin([1, 2, 3, 4])].sum()
                    / kouzh_zh.sum()*100), 0)
answer

In [ ]:
# подсчитываю доли и объединяю данные
share_by_region_22 = get_kouzh(2022)
share_by_region_24 = get_kouzh(2024)

# доля женщин, ухаживающих за детьми, в 2022 и 2024 году, сводная таблица
kouzh_combined = pd.merge(share_by_region_22, share_by_region_24, on='Номер региона', how='outer')
kouzh_combined['Номер показателя (ID)'] = 95853
# kouzh_combined['Описание'] = 'Рассчет «Горизонтальной России» по данным КОУЖ. Доля респондентов, в круг ежедневных занятий которых входит уход за детьми, от всех респондентов. Источник - Росстат: Комплексное наблюдение условий жизни населения.'
vlookup_kouzh(kouzh_combined, lookup, 'kouzh_code')
kouzh_combined.drop(kouzh_combined[kouzh_combined['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
kouzh_combined = kouzh_combined[['Номер региона (ОКАТО)', 
                   'Номер показателя (ID)', 
                   '2022', '2024']]

print(f"kouzh_combined has {kouzh_combined['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")
kouzh_combined.head()

# Обновление датасета «Если быть точным»

In [ ]:
# загружаю оригинальные данные «Если быть точным»
gsheetkey = "your/key"

sheet_name = 'problem_gender_inequality_all'
url=f'path/to/your/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
ebt = pd.read_excel(url,sheet_name=sheet_name)

In [ ]:
# выделяю из датасета «Если быть точным» только нужные мне показатели
ebt_use = ebt[ebt['Номер показателя (ID)'].isin([95799, 95789, 95790, 95853])]

# добавляю обозначение группы (м или ж) в описание показателя
mask = ebt_use['Номер показателя (ID)'] == 95789
ebt_use.loc[mask, 'Название показателя'] = ebt_use.loc[mask, 'Название показателя'] + ', мужчины'
mask = ebt_use['Номер показателя (ID)'] == 95790
ebt_use.loc[mask, 'Название показателя'] = ebt_use.loc[mask, 'Название показателя'] + ', женщины'

# преобразую строки в вещественные числа
ebt_use.loc[ebt_use['Название показателя'] == 'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', [i for i in range(2015, 2021, 2)]] = ebt_use.loc[ebt_use['Название показателя'] == 'Гендерный разрыв в оплате труда по группам занятий (в пользу мужчин)', [i for i in range(2015, 2021, 2)]].apply(lambda col: col.str.replace(',', '.', regex=False).astype(float))

ebt_use = ebt_use.drop(columns=['Группа показателей', 'Сегмент', 'Показывать', ' '])
    
ebt_use = vlookup_okato(ebt_use, lookup, 'ebt_use')
ebt_use.drop(ebt_use[ebt_use['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
# print(f"ebt_use has {ebt_use['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

ebt_use = ebt_use[['Номер региона (ОКАТО)',
                   'Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения',      
                   'Источник', 
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023, 2024]]
# Replace "н/д" with NaN in the year columns you want to fill
ebt_use = ebt_use.replace('н/д', np.nan)

In [ ]:
# добавляю данные о зарплатах и уходе за детьми
ebt_use = ebt_use.merge(
    salary_long[['Номер региона (ОКАТО)', 'Номер показателя (ID)', '2021', '2023']],
    on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'],
    how='left',
)
ebt_use[2021] = ebt_use[2021].fillna(ebt_use['2021'])
ebt_use[2023] = ebt_use[2023].fillna(ebt_use['2023'])
ebt_use.drop(columns=['2021', '2023'], inplace=True)

# дополняю описание показателя ЕБТ о гендерном разрыве в зарплатах
mask = ebt_use['Номер показателя (ID)'] == 95799
ebt_use.loc[mask, 'Описание'] = ebt_use.loc[mask, 'Описание'] + ' Данные за 2021 и 2023 год — рассчет «Горизонтальной России». Источник тот же.'

# добавляю данные из КОУЖ
ebt_use = ebt_use.merge(
    kouzh_combined,
    on=['Номер региона (ОКАТО)', 'Номер показателя (ID)'],
    how='left',
)
ebt_use[2022] = ebt_use[2022].fillna(ebt_use['2022'])
ebt_use[2024] = ebt_use[2024].fillna(ebt_use['2024'])
ebt_use.drop(columns=['2022', '2024'], inplace=True)
mask = ebt_use['Номер показателя (ID)'] == 95853
ebt_use.loc[mask, 'Описание'] = ebt_use.loc[mask, 'Описание'] + ' Данные за 2022 и 2024 год — рассчет «Горизонтальной России». Источник тот же.'

In [ ]:
# добавляю данные о средних зарплатах
# ожидаемая продолжительность жизни при рождении, женщины 2014-2026
# она отличается от данных ЕБТ, потому что Росстат задним числом пересчитал показатели
salary_long_all = salary_long.loc[salary_long['Номер показателя (ID)'] == 95916]
salary_long_all['Название показателя'] = 'Средняя заработная плата по группам занятий, все работники'
salary_long_all['Ед. измерения'] = 'руб.'
salary_long_all['Источник'] = 'ЕМИСС'
salary_long_all.columns = [int(c) if isinstance(c, str) and c.isdigit() else c for c in salary_long_all.columns]
# salary_long['Описание'] = ' '

salary_long_all = salary_long_all[['Номер региона (ОКАТО)', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                #    'Описание',
                #    2014, 2015, 2016, 2017, 2018,
                   2019, 
                # 2020, 
                   2021, 
                #    2022, 
                   2023]]

salary_long_all = okato_to_region(salary_long_all, lookup, 'datawrapper')
salary_long_all.sort_values(by='Название региона', ascending=True).head()

In [ ]:
ebt_use = pd.concat([ebt_use, salary_long_all], ignore_index=True)
ebt_use.tail()

# Данные для новых показателей

### Ожидаемая продолжительность жизни

In [ ]:
# ожидаемая продолжительность жизни при рождении, женщины 2014-2026
# она отличается от данных ЕБТ, потому что Росстат задним числом пересчитал показатели
life_exp = pd.read_excel('your/path/to/data/Ожидаемая продолжительность населения при рождении, женщины, по регионам, 2014-2023.xls')

life_exp.columns = ['Название региона'] + list(life_exp.iloc[1, 1:])
life_exp.columns = [life_exp.columns[0]] + [int(col) for col in life_exp.columns[1:]]
life_exp['Название региона'] = life_exp['Название региона'].str.strip()
life_exp = life_exp.drop(index=[0, 1])

# добавим номер показателя и название показателя как у ЕБТ
life_exp['Название показателя'] = 'Ожидаемая продолжительность жизни, женщины'
life_exp['Номер показателя (ID)'] = 95741
life_exp['Ед. измерения'] = 'год'
life_exp['Источник'] = 'ЕМИСС'
life_exp['Описание'] = 'Ожидаемая продолжительность жизни женщин при рождении'

life_exp = life_exp[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023]]

life_exp = vlookup_okato(life_exp, lookup, 'life_exp')
life_exp.drop(life_exp[life_exp['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [ ]:
ebt_use = pd.concat([ebt_use, life_exp], ignore_index=True)
ebt_use.tail()

### Доля женщин-потерпевших

In [ ]:
# женское население 2014-2025 по регионам
population_zh = pd.read_excel('your/path/to/data/Численность постоянного населения - женщин по возрасту на 1 января.xls')
population_zh.columns = ['Название региона'] + list(population_zh.iloc[1, 1:])
population_zh.columns = [population_zh.columns[0]] + [int(col) for col in population_zh.columns[1:]]
population_zh['Название региона'] = population_zh['Название региона'].str.strip()
population_zh = population_zh.drop(index=[0, 1])

population_zh = vlookup_okato(population_zh, lookup, 'population_zh')
population_zh.drop(population_zh[population_zh['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

population_zh.sort_values(2025, ascending=True).head()

In [ ]:
# Количество потерпевших-женщин (человек)
victims = pd.read_excel('your/path/to/data/Количество потерпевших-женщин, по регионам, 2014-2025.xls')
victims.columns = ['Название региона'] + list(victims.iloc[1, 1:])
victims.columns = [victims.columns[0]] + [int(col) for col in victims.columns[1:]]
victims = victims.drop(index=[0, 1, 2])

# добавим номер показателя и название показателя как у ЕБТ
# victims['Название показателя'] = 'Количество потерпевших-женщин'
# victims['Номер показателя (ID)'] = 95912
# victims['Ед. измерения'] = 'человек'

# чистим названия Забайкальского края, Крыма, Бурятии и Севастополя, сливаем данные из двух строк в одну 
victims['Название региона'] = victims['Название региона'].str.replace('\\', '', regex=False)
victims.iloc[93, 1:5] = victims.iloc[82, 1:5]
victims.iloc[46, 1:3] = victims.iloc[103, 1:3]
victims.iloc[51, 1:3] = victims.iloc[104, 1:3]
victims.iloc[91, 1:5] = victims.iloc[78, 1:5]

victims = victims.drop(index=[85, 106, 107, 81, 31]) # удаляем ненужные строки, включая ГУ МВД России по г.Санкт-Петербургу и Ленинградской области

# victims['Источник'] = 'ЕМИСС по данным МВД'
# victims['Описание'] = 'Количество потерпевших - женщин'

victims = victims[['Название региона', 
                  #  'Название показателя',
                  #  'Номер показателя (ID)', 
                  #  'Ед. измерения', 
                  #  'Источник',
                  #  'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023,
                   2024, 2025]]

victims = vlookup_okato(victims, lookup, 'victims')
victims.drop(victims[victims['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [ ]:
# подсчитываю долю женщин-потерпевших от женского населения по регионам
victims_population = victims.merge(
    population_zh, 
    on='Номер региона (ОКАТО)', 
    how='left', 
    suffixes=('_victims', '_population'))

for i in year_cols:
    for j in victims_population['Номер региона (ОКАТО)'].unique():
        if not victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_population'].eq(0).any():
            victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, str(i)] = (
                victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_victims'] /
                victims_population.loc[victims_population['Номер региона (ОКАТО)'] == j, f'{i}_population'] * 100000
                )
    victims_population = victims_population.drop(columns=[f'{i}_victims',f'{i}_population'])

# доля женщин-потерпевших от женского населения
victims_population = victims_population.rename(columns={'Название региона_victims': 'Название региона'})
victims_population = victims_population.drop(columns={'Название региона_population'})

victims_population.columns = [i for i in victims_population.columns[:2]] + [int(i) for i in victims_population.columns[2:]]

victims_population['Источник'] = 'Росстат: Российская Федерация без учета новых субъектов (с 01.01.2023) с учетом итогов Всероссийской переписи населения 2020 года, ЕМИСС по данным МВД (потерпевшие)'
victims_population['Описание'] = 'Расчеты «Горизонтальной России». Число потерпевших женщин на 100 000 женщин.'
victims_population['Ед. измерения'] = 'чел./100000'
victims_population['Номер показателя (ID)'] = 95914
victims_population['Название показателя'] = 'Потерпевшие женщины на 100 тыс. женщин'

victims_population = victims_population[['Номер региона (ОКАТО)',
                   'Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023,
                   2024, 2025]]

In [ ]:
ebt_use = pd.concat([ebt_use, victims_population], ignore_index=True)
ebt_use.tail()

### Количество изнасилований

In [ ]:
# Количество зарегистрированных преступлений (дел)
rape = pd.read_excel('your/path/to/data/Количество преступлений, зарегистрированных в отчетном периоде, изнасилование (ст. 131 УК РФ).xls')

In [ ]:
rape.columns = ['Название региона'] + list(rape.iloc[1, 1:])
rape.columns = [rape.columns[0]] + [int(col) for col in rape.columns[1:]]
rape = rape.drop(index=[0, 1])
# добавляю номер показателя и название показателя как у ЕБТ
rape['Название показателя'] = 'Количество изнасилований'
rape['Номер показателя (ID)'] = 95919
rape['Ед. измерения'] = 'единица'

# объединяю данные из двух строк в одну у Забайкальского края и Бурятии
rape['Название региона'] = rape['Название региона'].str.replace('\\', '', regex=False)
rape.iloc[92, 1:5] = rape.iloc[81, 1:5]
rape.iloc[90, 1:5] = rape.iloc[77, 1:5]

rape = rape.drop(index=[80, 84, 30]) # удаляем ненужные строки, включая ГУ МВД России по г.Санкт-Петербургу и Ленинградской области
for year in year_cols:
    s = pd.to_numeric(rape[year], errors='coerce')
    rape[year] = s.mask(s == 0.0, 1.0) * 7.5      
rape['Источник'] = 'Расчет «Горизонтальной России» на данных МВД'
rape['Описание'] = 'Предполагаемое фактическое количество изнасилований. Количество зарегистрированных изнасилований (ст. 131 УК РФ) было умножено на коэффициент латентности 7,5.'

rape = rape[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023,
                   2024, 2025]]

rape = vlookup_okato(rape, lookup, 'rape')
rape.drop(rape[rape['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [ ]:
# ЧИСЛЕННОСТЬ МУЖЧИН по регионам, 2014-2024
#   2014-2021: Chisl_polvozr_01-01-YYYY_VPN-2020.xlsx  (лист 'М_Г+С' — мужчины, город+село)
#              кол.0 = регион, кол.1 = «Всего» (все мужчины, любой возраст)
#   2022-2024: Chisl_RF_01-01-2022-01-01-2024__1_.xls  (лист на каждый год)
#              кол.0 = регион, кол.2 = мужчины (всё население)
# Знаменатель = всё мужское население (единственный показатель, доступный
# во всех девяти файлах: файл 2022-2024 не содержит разбивки по возрасту).
POP_DIR = 'your/path/to/data/численность постоянного населения'

def _clean_region(s):
    return (s.astype(str)
             .str.replace('\xa0', ' ', regex=False)   # неразрывные пробелы
             .str.replace('–', '-', regex=False)       # длинное тире -> дефис
             .str.replace(r'\s+', ' ', regex=True)
             .str.strip())

male_by_year = {}

# 2014-2021: погодовые переписные файлы
for y in range(2014, 2022):
    d = pd.read_excel(f'{POP_DIR}/Chisl_polvozr_01-01-{y}_VPN-2020.xlsx',
                      sheet_name='М_Г+С', header=None)
    d = d.iloc[8:, [0, 1]].copy()                      # данные с 9-й строки: регион, «Всего»
    d.columns = ['Название региона', y]
    d['Название региона'] = _clean_region(d['Название региона'])
    d[y] = pd.to_numeric(d[y], errors='coerce')
    male_by_year[y] = d.dropna(subset=[y]).set_index('Название региона')[y]

# 2022-2024: сводный файл, лист на каждый год
rf_path = f'{POP_DIR}/Chisl_RF_01-01-2022-01-01-2024 (1).xls'
for ys in ['2022', '2023', '2024']:
    d = pd.read_excel(rf_path, sheet_name=ys, header=None)
    d = d.iloc[6:, [0, 2]].copy()                      # данные с 7-й строки: регион, мужчины
    d.columns = ['Название региона', int(ys)]
    d['Название региона'] = _clean_region(d['Название региона'])
    d[int(ys)] = pd.to_numeric(d[int(ys)], errors='coerce')
    male_by_year[int(ys)] = d.dropna(subset=[int(ys)]).set_index('Название региона')[int(ys)]

# собираем в одну таблицу: регион × год (2014…2024)
population_m = pd.concat(male_by_year.values(), axis=1)
population_m.columns = list(male_by_year.keys())
population_m = population_m.reset_index().rename(columns={'index': 'Название региона'})

population_m = vlookup_okato(population_m, lookup, 'population_m')
population_m.drop(population_m[population_m['Номер региона (ОКАТО)'] == '0'].index, inplace=True)

In [ ]:
# ИЗНАСИЛОВАНИЯ на 100 000 мужчин по регионам
#   rapes — датафрейм с числом изнасилований («Количество изнасилований»),
#           той же структуры, что victims (Название региона, ОКАТО, годовые колонки-int)
rape_rate = rape.merge(
    population_m,
    on='Номер региона (ОКАТО)',
    how='left',
    suffixes=('_rapes', '_population'))

for i in range(2014, 2025):
    for j in rape_rate['Номер региона (ОКАТО)'].unique():
        if not rape_rate.loc[rape_rate['Номер региона (ОКАТО)'] == j, f'{i}_population'].eq(0).any():
            rape_rate.loc[rape_rate['Номер региона (ОКАТО)'] == j, str(i)] = (
                rape_rate.loc[rape_rate['Номер региона (ОКАТО)'] == j, f'{i}_rapes'] /
                rape_rate.loc[rape_rate['Номер региона (ОКАТО)'] == j, f'{i}_population'] * 100000
                )
    rape_rate = rape_rate.drop(columns=[f'{i}_rapes', f'{i}_population'])

rape_rate = rape_rate.rename(columns={'Название региона_rapes': 'Название региона'})
rape_rate = rape_rate.drop(columns={'Название региона_population'})
rape_rate = rape_rate.drop(columns=[2025.0], errors='ignore')
rape_rate.columns = [int(c) if str(c).lstrip('-').isdigit() else c for c in rape_rate.columns]

In [ ]:
# посчитаю скользящее среднее с 2021 по 2023 годы
for y in [2021, 2022, 2023]:
    rape_rate[y] = pd.to_numeric(rape_rate[y], errors='coerce')
rape_rate[2023] = rape_rate[[2021, 2022, 2023]].mean(axis=1)

rape_rate['Источник'] = 'Росстат (численность), ЕМИСС по данным МВД (изнасилования)'
rape_rate['Описание'] = 'Расчеты «Горизонтальной России». Усредненное число изнасилований на 100 000 мужчин за 2021-2023 годы.'
rape_rate['Ед. измерения'] = 'чел./100000'
rape_rate['Номер показателя (ID)'] = 95921
rape_rate['Название показателя'] = 'Изнасилования на 100 тыс. мужчин'

rape_rate = rape_rate[['Номер региона (ОКАТО)',
                   'Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                #    2014, 2015, 2016, 2017, 2018,
                #    2019, 2020, 2021, 2022, 
                   2023,
                #    2024, 2025
                   ]]

In [ ]:
ebt_use = pd.concat([ebt_use, rape, rape_rate], ignore_index=True)
ebt_use.tail()

### Подростковые беременности и роды

In [ ]:
# Возрастные коэффициенты рождаемости (15-19 лет), 2014-2023
teen_birth = pd.read_excel('your/path/to/data/Возрастные коэффициенты рождаемости (15-17 лет).xls')

teen_birth.columns = ['Название региона'] + list(teen_birth.iloc[1, 1:])
teen_birth.columns = [teen_birth.columns[0]] + [int(col) for col in teen_birth.columns[1:]]
teen_birth['Название региона'] = teen_birth['Название региона'].str.strip()
teen_birth = teen_birth.drop(index=[0, 1, 27, 74]) # удаляю пустые строки, а также Архангельскую область (без АО) и Тюменскую область (без АО), они дублируются

# добавлю номер показателя и название показателя как у ЕБТ
teen_birth['Название показателя'] = 'Рождаемость среди подростков (15-17 лет)'
teen_birth['Номер показателя (ID)'] = 95920
teen_birth['Ед. измерения'] = 'чел./1000'
teen_birth['Источник'] = 'Росстат'
teen_birth['Описание'] = 'Возрастные коэффициенты рождаемости (15-17 лет)'

teen_birth = teen_birth[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2014, 2015, 2016, 2017, 2018,
                   2019, 2020, 2021, 2022, 2023]]


teen_birth = vlookup_okato(teen_birth, lookup, 'teen_birth')
teen_birth.drop(teen_birth[teen_birth['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
# teen_birth.sort_values(by='Название региона', ascending=True).head()


In [ ]:
print(f"teen_birth has {teen_birth['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

In [ ]:
ebt_use = pd.concat([ebt_use, teen_birth], ignore_index=True)
ebt_use.tail()

### Материнская смертность

In [ ]:
# материнская смертность на 100 тыс. новорожденных, женщины 2014-2026
# она отличается от данных ЕБТ, потому что Росстат задним числом пересчитал показатели
mother_death = pd.read_excel('/your/path/to/data/Материнская смертность на 100 тыс. родившихся.xls')
mother_death.columns = ['Название региона'] + list(mother_death.iloc[1, 1:])
mother_death.columns = [mother_death.columns[0]] + [int(col) for col in mother_death.columns[1:]]
mother_death['Название региона'] = mother_death['Название региона'].str.strip()
mother_death.loc[2, 2024] = mother_death.loc[3, 2024]
mother_death = mother_death.drop(index=[0, 1, 3]) # удаляю пустые строки и РФ после 2023

# посчитаю скользящее среднее с 2022 по 2024 годы
for y in [2022, 2023, 2024]:
    mother_death[y] = pd.to_numeric(mother_death[y], errors='coerce')
mother_death[2023] = mother_death[[2022, 2023, 2024]].mean(axis=1)

# добавим номер показателя и название показателя как у ЕБТ
mother_death['Название показателя'] = 'Коэффициент материнской смертности'
mother_death['Номер показателя (ID)'] = 95922
mother_death['Ед. измерения'] = 'чел./100000'
mother_death['Источник'] = 'Рассчет «Горизонтальной России» по данным ЕМИСС'
mother_death['Описание'] = 'Простое скользящее среднее коэффициентов за 2022-2024 годы. Росстат подсчитывает коэффициент материнской смертности как число умерших женщин от осложнений беременности, родов и послеродового периода. Рассчитывается на 100 тысяч родившихся живыми.'

mother_death = mother_death[['Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                  #  2014, 2015, 2016, 
                  #  2017, 2018,2019, 
                  #  2020, 2021, 2022, 
                   2023, 
                  #  2024
                   ]]

mother_death = vlookup_okato(mother_death, lookup, 'mother_death')
mother_death.drop(mother_death[mother_death['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
print(f"mother_death has {mother_death['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

In [ ]:
ebt_use = pd.concat([ebt_use, mother_death], ignore_index=True)
ebt_use.tail()

### Доступность продуктов

In [ ]:
# стоимость продуктовой корзины в октябре (во время опроса о средних зп) с 2014 по 2026 гг.
groceries = pd.read_excel('your/path/to/data/Стоимость фиксированного набора потребительских товаров и услуг по регионам, 2014-2026.xls')

groceries.columns = ['Название региона'] + list(groceries.iloc[1, 1:])
groceries.columns = [groceries.columns[0]] + [int(col) for col in groceries.columns[1:]]
groceries['Название региона'] = groceries['Название региона'].str.strip()
groceries.iloc[3, 10:] = groceries.iloc[4, 10:]
groceries = groceries.drop(index=[0, 1, 2, 4, 29, 77]) # удаляю ненужные или дублирующиеся строки, пр., "Архангельская область (без АО)" и "Тюменская область (без АО)"

groceries = vlookup_okato(groceries, lookup, 'groceries')
groceries.drop(groceries[groceries['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
print(f"groceries has {groceries['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

In [ ]:
# доля корзины от средней зарплаты женщин
salary_long_w = ebt_use[ebt_use['Номер показателя (ID)'] == 95790] # выделяю только среднюю зарплату женщин, которую буду использовать в подсчетах
year_columns = [2015, 2017, 2019, 2021, 2023]

# salary_long_w.loc[:, 'Номер региона (ОКАТО)'] = salary_long_w.loc[:, 'Номер региона (ОКАТО)'].str.replace('.0', '', regex=False)
for i in year_columns:
    salary_long_w.loc[:, i] = pd.to_numeric(salary_long_w.loc[:, i], errors='coerce')

# соединяю два датасета
affordability = salary_long_w[['Название региона', 'Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]].merge(
    groceries[['Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]],
    on=['Номер региона (ОКАТО)'],
    suffixes=('_wages', '_groceries'),
    how='left'
)

affordability['Номер показателя (ID)'] = 95913
affordability['Название показателя'] = 'Доля товаров и услуг от зарплаты'
affordability['Ед. измерения'] = '%'
affordability['Описание'] = 'Расчет «Горизонтальной России». Средняя  начисленная зарплата мужчин и женщин по группам занятий' \
                         'и субъектам Российской Федерации за октябрь отчетного года.' \
                         'Источник - Росстат: Сведения о заработной плате работников в организациях по категориям персонала и профессиональным группам работников.' \
                         '\nСтоимость продуктовой корзины. Источник - Росстат: Стоимость фиксированного набора потребительских товаров и услуг.'
affordability['Источник'] = 'Источник данных о зарплате женщин — Росстат, о стоимости продуктовой корзины — ЕМИСС, Росстат.'

# добавляю рассчет доли корзины от средней зарплаты женщин в процентах 2014-2023
for i in year_columns:
    affordability[i] = affordability[f'{i}_groceries'] / affordability[f'{i}_wages'] * 100

affordability = affordability.drop(columns=[f'{i}_wages' for i in year_columns] + [f'{i}_groceries' for i in year_columns])
affordability.sort_values(by='Название региона', ascending=True).head()
print(f"affordability has {affordability['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")


### Реальная зарплата

In [ ]:
# индексе потребительских цен от года к году по регионам, 2014-2018
cpi = pd.read_excel('your/path/to/data/Индексы потребительских цен на товары и услуги, 2014-2024.xls')
cpi.columns = ['Название региона'] + list(cpi.iloc[1, 1:])
cpi = cpi.drop(columns=cpi.columns[1])
cpi.columns = [cpi.columns[0]] + [int(col) for col in cpi.columns[1:]]
cpi['Название региона'] = cpi['Название региона'].str.strip()
cpi.iloc[4, -2:] = cpi.iloc[5, -2:]
cpi = cpi.drop(index=[0, 1, 2, 3, 5])

cpi = vlookup_okato(cpi, lookup, 'cpi')
cpi.drop(cpi[cpi['Номер региона (ОКАТО)'] == '0'].index, inplace=True)
print(f"cpi has {cpi['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")
# cpi

In [ ]:
# считаю реальную зп женщин с помощью индекса потребительских цен
# соединяю два датасета
odd_years = [2015, 2017, 2019, 2021, 2023]

real_wages = salary_long_w[['Название региона', 'Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]].merge(
    cpi[['Номер региона (ОКАТО)', 2015, 2017, 2019, 2021, 2023]],
    on=['Номер региона (ОКАТО)'],
    suffixes=('_wages', '_cpi'),
    how='left'
)

real_wages['Номер показателя (ID)'] = 95915
real_wages['Название показателя'] = 'Реальная средняя начисленная зарплата женщин'
real_wages['Ед. измерения'] = 'руб.'
real_wages['Описание'] = 'Расчет «Горизонтальной России». Средняя  начисленная зарплата мужчин и женщин по группам занятий. ' \
                         'и субъектам Российской Федерации за октябрь отчетного года. ' \
                         'Источник - Росстат: Сведения о заработной плате работников в организациях по категориям персонала и профессиональным группам работников. ' \
                         '\nИндексы потребительских цен на товары и услуги к соответствующему периоду предыдущего года, данные за каждый октябрь. ' \
                         'Оккупированные территории «ДНР», «ЛНР», Запорожской и Херсонской областей не включены в рассчеты. Источник - Росстат. '
real_wages['Источник'] = 'Источник данных о зарплате женщин и об индексе потребительских цен — Росстат.'

for i in odd_years:
    real_wages.loc[:, i] = real_wages.loc[:, f'{i}_wages'] / (real_wages.loc[:, f'{i}_cpi'] / 100)
    drop_cols = [f'{i}_wages', f'{i}_cpi']
    real_wages = real_wages.drop(columns=drop_cols)

for j in odd_years:
    real_wages[j] = pd.to_numeric(real_wages[j], errors='coerce').round(0)

# real_wages.sort_values(by='Название региона', ascending=True).head()
print(f"real_wages has {real_wages['Номер региона (ОКАТО)'].isna().sum()} missing values in 'Номер региона (ОКАТО)' column.")

In [ ]:
ebt_use = pd.concat([ebt_use, affordability, real_wages], ignore_index=True)
ebt_use.tail()

### Доля женщин без основного общего образования

In [ ]:
filepath = 'your/path/to/data/КДУ (Выборочное наблюдения качества и доступности услуг)/'

def calculate_share(group):
    """Доля женщин 15–60 лет без основного общего образования (I01_08 == 1)."""
    group_working_age = group[(group['I01_02'] >= 15) & (group['I01_02'] <= 60)]
    total_w = group_working_age['KVZV'].sum()
    if total_w == 0:
        return None
    no_edu_w = group_working_age[group_working_age['I01_08'] == 1]['KVZV'].sum()
    return round(no_edu_w / total_w * 100, 0)

def get_kdu(year):
    if year > 2019:
        kdu = pd.read_csv(
            f'{filepath}{year}/КДУ_{year}_Датасет_по_индивидуальным_данным_основной.csv',
            sep=';'
        )
    else:
        kdu = pd.read_csv(
            f'{filepath}{year}/КДУ_{year}_Датасет_по_индивидуальным_данным.csv',
            sep=';'
        )

    # Выделяю только женщин трудоспособного возраста (15–60 лет)
    kdu_zh = kdu[(kdu['I01_01'] == 2) &
                 (kdu['I01_02'] >= 15) &
                 (kdu['I01_02'] <= 60)].copy()

    # Доля без основного общего образования по регионам
    share_by_region = (
        kdu_zh.groupby('I00_02')
        .apply(calculate_share, include_groups=False)
        .reset_index()
        .rename(columns={0: year, 'I00_02': 'Номер региона'})
    )

    # Добавляю строку «Россия» (код 100)
    total_w = kdu_zh['KVZV'].sum()
    no_edu_total = kdu_zh[kdu_zh['I01_08'] == 1]['KVZV'].sum()
    russia_share = round(no_edu_total / total_w * 100, 0)

    share_by_region = pd.concat([
        share_by_region,
        pd.DataFrame({'Номер региона': [100], year: [russia_share]})
    ], ignore_index=True)

    return share_by_region[['Номер региона', year]]

In [ ]:
# Загружаю данные по каждому году
kdu_2017 = get_kdu(2017)
kdu_2019 = get_kdu(2019)
kdu_2021 = get_kdu(2021)
kdu_2023 = get_kdu(2023)

# Объединяю все годы
kdu_combined = (
    kdu_2017
    .merge(kdu_2019, on='Номер региона', how='outer')
    .merge(kdu_2021, on='Номер региона', how='outer')
    .merge(kdu_2023, on='Номер региона', how='outer')
)

# добавлю номер показателя и название показателя как у ЕБТ
kdu_combined['Название показателя'] = 'Доля необразованных женщин'
kdu_combined['Номер показателя (ID)'] = 95918
kdu_combined['Ед. измерения'] = '%'
kdu_combined['Источник'] = 'Росстат, КДУ (Выборочное наблюдения качества и доступности услуг)'
kdu_combined['Описание'] = 'Доля женщин 15-60 лет без основного общего образования'

# Добавляю ID показателя и убираем строки без ОКАТО
vlookup_kouzh(kdu_combined, lookup, 'kdu')

In [ ]:
kdu_combined.drop(
    kdu_combined[kdu_combined['Номер региона (ОКАТО)'] == '0'].index,
    inplace=True
)

kdu_combined = okato_to_region(kdu_combined, lookup, 'datawrapper')

kdu_combined.head()

kdu_combined = kdu_combined[['Номер региона (ОКАТО)',
                   'Название региона', 
                   'Название показателя',
                   'Номер показателя (ID)', 
                   'Ед. измерения', 
                   'Источник',
                   'Описание',
                   2017, 2019, 2021, 2023]]


In [ ]:
ebt_use = pd.concat([ebt_use, kdu_combined], ignore_index=True)
ebt_use.tail()

# Обработка итогового датасета

In [ ]:
ebt_use[ebt_use['Номер региона (ОКАТО)'].isna()]

In [ ]:
ebt_use = ebt_use.drop(columns='Название региона')
ebt_use = okato_to_region(ebt_use, lookup, 'datawrapper')
# ebt_use = ebt_use.rename(columns={'datawrapper':'Название региона'})

missing = sorted(set(ebt_use.loc[ebt_use['Название региона'].isna(), 'Номер региона (ОКАТО)']))
print('Missing OKATO codes:', missing)
ebt_use.head()

In [ ]:
ebt_use['Номер региона (ОКАТО)'] = (
    pd.to_numeric(ebt_use['Номер региона (ОКАТО)'].replace('nan', np.nan), errors='coerce')
    .astype('Int64')
)

# заполняю пропуски данными прошлого года для показателя "Забота о других" (ID 95853)
mask = ebt_use['Номер показателя (ID)'] == 95853
ebt_use.loc[mask, [y for y in year_cols]] = ebt_use.loc[mask, [y for y in year_cols]].ffill(axis=1)
ebt_use.loc[mask, [y for y in year_cols]]

# Считаем рейтинг 7х7

In [ ]:
indicators = ebt_use[['Название показателя', 'Номер показателя (ID)']].drop_duplicates()
print(indicators)

In [ ]:
# how many indicators have 0 value in 2023
ebt_use[ebt_use[2023] == 0]

In [ ]:
# нормализую данные

# convert year number in column names to numeric
for i in ebt_use.columns[7:]:
    ebt_use.columns = [
        int(col.strip()) if isinstance(col, str) and col.strip().isdigit() else col
        for col in ebt_use.columns
    ]

ebt_use['Номер показателя (ID)'] = ebt_use['Номер показателя (ID)'].astype(str)

# показатели, где меньше — лучше:
# гендерный разрыв в зарплатах ('95799'), Время на уход за детьми ('95853'),
# кол-во женщин-потерпевших на 100 тыс. женщин ('95914'), кол-во изнасилований на 100 тыс. мужчин ('95921') 
# рождаемость среди подростков ('95920'), доля необразованных женщин ('95918')
# и доля расходов на питание ('95913'), Коэффициент материнской смертности ('95922')
invert_ids = ['95799', '95853', '95914', '95921', '95920', '95918', '95913', '95922']

# удаляю из датасета показатели, не учитывающиеся в рейтинге: 
# Запрет на пропаганду абортов ('95911'), среднюю зарплату мужчин ('95789'), женщин ('95790'),
# реальную среднюю зарплату женщин ('95915'), кол-во изнасилований ('95919')
# оставляю среднюю зарплату всех работников ('95916') для корректирующего показателя
ebt_use_2 = ebt_use[~ebt_use['Номер показателя (ID)'].isin(['95911', '95789', '95790', '95915', '95919'])]
odd_years = [2023] 

# удаляю оккупированные и аннексированные украинские регионы:
# «ДНР», Запорожскую область, «ЛНР», Херсонскую область, Крым и Севастополь
# а также Архангельскую и Тюменскую области с автономными округами и РФ. 
#  удаляю регионы СКФО и Адыгею из-за ненадежности данных
ebt_use_2 = ebt_use_2[~ebt_use_2['Номер региона (ОКАТО)'].isin([21000000000, 23000000000, 
                                                              43000000000, 74000000000, 
                                                              35000000, 67000000, 
                                                              11000000, 71000000, 
                                                              100000000000, 82000000, 
                                                              26000000, 83000000, 
                                                              90000000, 96000000, 
                                                              91000000, 7000000, 
                                                              79000000])]

def normalize_variable(df, invert):
    result = df.copy()
    for year in odd_years:
        col = pd.to_numeric(df[year], errors='coerce')
        x_min, x_max = col.min(), col.max()
        normalized = (col - x_min) / (x_max - x_min)
        result[year] = 1 - normalized if invert else normalized
    return result

normalized_parts = []
for var_id, group in ebt_use_2.groupby('Номер показателя (ID)'):
    invert = var_id in invert_ids
    normalized_parts.append(normalize_variable(group, invert=invert))

ebt_normalized = pd.concat(normalized_parts).sort_index()

In [ ]:
indicators = ebt_normalized[['Название показателя', 'Номер показателя (ID)']].drop_duplicates()
print(indicators)

In [ ]:
# показатели общего уровня зарплат ('95916') и гендерного разрыва в зарплатах ('95799') учитываются в рейтинге с весом 0.5, остальные — с весом 1.0
half_weight_ids = {'95799', '95916'}
ebt_normalized['_weight'] = np.where(
    ebt_normalized['Номер показателя (ID)'].astype(str).isin(half_weight_ids), 0.5, 1.0)

for year in odd_years:
    val = pd.to_numeric(ebt_normalized[year], errors='coerce')
    w   = ebt_normalized['_weight'].where(val.notna())   # вес не считается там, где значение отсутствует

    num = (val * ebt_normalized['_weight']).groupby(ebt_normalized['Номер региона (ОКАТО)']).sum()
    den = w.groupby(ebt_normalized['Номер региона (ОКАТО)']).sum()
    means = num / den

    ebt_normalized[f'Рейтинг 7х7_{year}'] = ebt_normalized['Номер региона (ОКАТО)'].map(means)

ebt_normalized = ebt_normalized.drop(columns='_weight')

In [ ]:
import jenkspy

rating_cols = ['Рейтинг 7х7_2023']
labels = ['Очень плохо', 'Плохо', 'Нормально', 'Хорошо', 'Отлично']
for col in rating_cols:
    breaks = jenkspy.jenks_breaks(ebt_normalized[col].astype(float).dropna(), n_classes=5)
    breaks[0] = 0
    ebt_normalized[f'{col}_оценка'] = pd.cut(ebt_normalized[col], bins=breaks, labels=labels, include_lowest=True, ordered=False)

# Сохраняю итоговый рейтинг

In [ ]:
ebt_use = ebt_use.merge(ebt_normalized[['Номер региона (ОКАТО)', 'Рейтинг 7х7_2023', 'Рейтинг 7х7_2023_оценка']], 
                                on=['Номер региона (ОКАТО)'], 
                                how='left')

In [ ]:
ebt_use['Номер региона (ОКАТО)']

In [ ]:
unclear_okato = [82000000, 26000000, 83000000,
                 90000000, 96000000, 91000000,
                 7000000,  79000000]

id_to_col = {
    '95915': 'real avg. salary women',
    '95799': 'gender pay gap',
    '95741': 'life expectancy',
    '95920': "teens' birth rate",
    '95922': 'maternal mortality rate',
    '95919': 'no. of rape',
    '95918': 'uneducated',
    '95913': 'affordability',
    '95853': 'everyday childcare',
    '95921': 'share of rape',
    '95914': 'share of victims',
}

year_col = 2023 if 2023 in ebt_use.columns else '2023'

# широкая таблица значений показателей (из ebt_use — включает ВСЕ регионы)
vals = ebt_use.copy()
vals['Номер показателя (ID)'] = vals['Номер показателя (ID)'].astype(str)   # ID к строке (устойчиво до/после ячейки 80)
vals = vals[vals['Номер показателя (ID)'].isin(id_to_col)]
pivot_vals = (
    vals.pivot_table(index='Номер региона (ОКАТО)',
                     columns='Номер показателя (ID)',
                     values=year_col,
                     aggfunc='first')
        .rename(columns=id_to_col)
)

# оценка и числовой рейтинг (из ebt_normalized — только оценённые регионы)
rating = (
    ebt_normalized.groupby('Номер региона (ОКАТО)')
    .agg(**{'rating 2023':           ('Рейтинг 7х7_2023_оценка', 'first'),
            'rating 2023 numerical': ('Рейтинг 7х7_2023',        'first')})
)

# добавляю «Неясно»-регионы в индекс рейтинга и помечаю их
extra = pd.Index(unclear_okato, name=rating.index.name).astype(rating.index.dtype)
rating = rating.reindex(rating.index.union(extra))

rating['rating 2023'] = rating['rating 2023'].astype('object')   # снимаем ограничения категории, чтобы вписать «Неясно»
mask_unclear = rating.index.isin(unclear_okato)
rating.loc[mask_unclear, 'rating 2023'] = 'Неясно'
rating.loc[mask_unclear, 'rating 2023 numerical'] = np.nan       # у «Неясно» числового рейтинга нет

print(rating['rating 2023'].value_counts(dropna=False))

region_name = (
    ebt_use.dropna(subset=['Название региона'])
           .drop_duplicates('Номер региона (ОКАТО)')
           .set_index('Номер региона (ОКАТО)')['Название региона']
           .rename('region name')
)

summary = rating.join(region_name).join(pivot_vals)

for c in id_to_col.values():
    if c in summary.columns:
        summary[c] = pd.to_numeric(summary[c], errors='coerce')

col_order = ['region name', 'rating 2023', 'rating 2023 numerical',
             'real avg. salary women', 'gender pay gap', 'life expectancy',
             "teens' birth rate", 'maternal mortality rate', 'no. of rape',
             'uneducated', 'affordability', 'everyday childcare',
             'share of rape', 'share of victims']

summary = summary.reset_index()
summary = summary[[c for c in col_order if c in summary.columns]]
summary['rating 2023 numerical'] = summary['rating 2023 numerical'] * 100
summary = summary.sort_values('rating 2023 numerical', ascending=False,
                              na_position='last').reset_index(drop=True)

In [ ]:
xlsx_path = path_to_save + 'рейтинг таблица и карта 3.xlsx'

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='рейтинг таблица и карта 3', index=False)
    ebt_use.to_excel(writer, sheet_name='ebt_use_step_6.4', index=False)

print('Сохранено:', xlsx_path)